# Lab 2 Part 1: evidence-first Kaggle run (v7)

Import this one file into Kaggle and run it with a nominal 16-GB GPU and Internet enabled. All executable source is embedded; course PDFs download from pinned URLs with SHA-256 checks. The notebook creates its helper files automatically and uses an isolated pinned Python environment. No repository clone, extra Python upload or Hugging Face token is required for the public models. Its source is checked to stay below Kaggle's 1-MB saving limit.

Start in a new Kaggle notebook with a GPU and Internet enabled, then run all cells in order. No saved run folder, recovery notebook, previous ZIP, separate Python file, repository checkout or Hugging Face token is needed. The first cell unpacks its own verified executable source, creates its isolated Python environment and downloads the three hash-checked course PDFs. On a fresh session it creates everything it needs automatically. Keep course work and saved outputs private.

The final system uses hybrid dense/BM25/CLIP retrieval, reranking, table grids, exact chart references, source trust and per-conversation memory. It compares against a dense-only baseline on a separate 25-question candidate set. The model never receives the candidate reference answers. Both systems use the same pinned Qwen2-VL-7B NF4 model with float16 vision on one nominal 16-GB GPU. T4 x2 is supported, but only one card is used; VRAM is not pooled.

Answers begin with natural JSON; malformed answers retry with evidence-bounded constrained decoding. Only supplied citation/figure labels are allowed, and an empty figure evidence list requires an empty figure-reference list. The citation judge always uses exact source keys and real boolean values, even when answer generation uses natural JSON. GPU preflight exercises both chart answering and citation judging before the long stages. Required validation is not bypassed and missing scores are not invented.

The earlier optional draft editor is disabled. It was not required by the rubric, repeatedly failed its own checks and consumed extra model calls. Required citation evaluation and the baseline comparison are still performed. This changes the measured profile, so old answer checkpoints are not reused or claimed as new results. No particular leaderboard score or runtime is guaranteed. The 90-minute requirement is checked against the actual elapsed time, including setup, rather than asserted in advance.

Attach the competition data if available. Its Questions.csv is selected automatically; otherwise the embedded official practice file is used. Questions are not rewritten or reordered. Complete predictions are validated and published as submission_part1_v7.csv before the separate golden evaluation. Results and per-item checkpoints are saved as they complete; diagnostics and a ZIP preserve available evidence if a later stage fails. On a new session, unsaved files from an old session are not needed or recovered.

Only Part 1 runs. Part 2 training and Part 3 are not repeated. A successful CPU smoke checks packaging/control flow and tiny model interfaces, not full pretrained CUDA execution or answer quality. Real groundedness ratings, team reference review, report/team records, a tag and external Kaggle/Canvas submission remain separate.

For a background run, use Save Version > Save & Run All after setting the accelerator and Internet. Download the new CSV, Results ZIP and executed notebook from Kaggle's Output panel when finished.

### What changed from the completed v6 run

The model now reads short source observations before writing its answer and
choosing a status. Its prompt contains only earlier user turns, not old assistant
claims that could be wrong. Table rows retain their headers without duplicating
the entire numeric matrix. Exact cited object metadata supplies matching figure
labels. Assumptions remain chat-only, and the original PDFs are never edited.

This notebook does not contain the previous run's answers or a guaranteed score.
It does not read candidate reference answers during generation. The software
checks cannot establish factual correctness: inspect the measured evaluation and
new Kaggle score after running. v6 remains a separate preserved notebook. This
revision still requires the full GPU run; CPU smoke results are not GPU results.

In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, subprocess, json, hashlib, base64, io, stat, zipfile, time
NOTEBOOK_STARTED = time.perf_counter()
SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
"""Kaggle-specific secrets, temporary caches and persistent result delivery."""
import hashlib
import os
from pathlib import Path
import shutil
import zipfile


def configure_runtime(working=None, scratch=None, environ=None, secret_loader=None, smoke_only=False,
                      require_private_artifacts=True):
    """Configure before ML imports. Never prompt for a token during a batch run."""
    if smoke_only:
        return {"mode": "CPU smoke only; no Kaggle GPU execution is certified"}
    environ = os.environ if environ is None else environ
    working = Path(working or "/kaggle/working").resolve()
    scratch = Path(scratch or "/kaggle/temp/lab2-runtime").resolve()
    if not working.is_dir():
        raise RuntimeError("Import this notebook into Kaggle, select a GPU and enable Internet before running it.")
    if scratch == working or working in scratch.parents:
        raise ValueError("Large caches and environments must be outside Kaggle's saved output directory.")
    token = environ.get("HF_TOKEN", "").strip()
    if not token and require_private_artifacts:
        try:
            if secret_loader is None:
                from kaggle_secrets import UserSecretsClient
                secret_loader = UserSecretsClient().get_secret
            token = secret_loader("HF_TOKEN").strip()
        except Exception:
            raise RuntimeError("In Kaggle, open Add-ons > Secrets, add HF_TOKEN with your Hugging Face read token, "
                               "enable it for this notebook, then rerun. Never paste the token into a cell.") from None
    if not token and require_private_artifacts:
        raise RuntimeError("The HF_TOKEN secret is empty. Set a read token with access to your private Part 2 artifacts.")
    scratch.mkdir(parents=True, exist_ok=True)
    if token:
        environ["HF_TOKEN"] = token
    # A T4 x2 runtime still uses one card. Two devices are not one 32-GB GPU.
    visible = environ.get("CUDA_VISIBLE_DEVICES", "0") or "0"
    environ["CUDA_VISIBLE_DEVICES"] = visible.split(",")[0]
    paths = {
        "HF_HOME": scratch / "huggingface",
        "HF_HUB_CACHE": scratch / "huggingface/hub",
        "HF_DATASETS_CACHE": scratch / "datasets",
        "TORCH_HOME": scratch / "torch",
        "XDG_CACHE_HOME": scratch / "cache",
        "UV_CACHE_DIR": scratch / "uv-cache",
        "UV_PYTHON_INSTALL_DIR": scratch / "uv-python",
        "LAB2_ENV_DIR": scratch / "environments",
        "LAB2_BOOTSTRAP_DIR": scratch / "bootstrap",
    }
    for name, path in paths.items():
        environ[name] = str(path)
    environ["LAB2_KAGGLE_WORKING"] = str(working)
    environ["LAB2_STANDALONE_PARENT"] = str(working)
    # Return paths and hardware selection, never the secret.
    return {"working": str(working), "temporary_cache": str(scratch),
            "visible_gpu": environ["CUDA_VISIBLE_DEVICES"], "secret_loaded": bool(token)}


def publish_zip(archive, working=None, display_link=True):
    """Keep the exact validated archive at the top of Kaggle's Output panel."""
    archive = Path(archive).resolve()
    working = Path(working or "/kaggle/working").resolve()
    if not working.is_dir() or not archive.is_file() or archive.suffix.lower() != ".zip":
        raise FileNotFoundError("A complete result ZIP and an existing output directory are required.")
    with zipfile.ZipFile(archive) as bundle:
        if bundle.testzip() is not None:
            raise ValueError("Result ZIP failed its integrity check; it was not published.")
    def checksum(path):
        value = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(2**20), b""):
                value.update(block)
        return value.hexdigest()
    destination = working / archive.name
    expected = checksum(archive)
    if destination.is_symlink():
        raise FileExistsError("The output destination is a symlink and was preserved.")
    if destination != archive:
        if destination.exists():
            if checksum(destination) != expected:
                raise FileExistsError("A different same-named result ZIP exists and was preserved.")
        else:
            if shutil.disk_usage(working).free < archive.stat().st_size + 64 * 2**20:
                raise RuntimeError("Not enough space for the Output-panel copy. The original ZIP is preserved.")
            shutil.copy2(archive, destination)
    if checksum(destination) != expected:
        raise ValueError("Output copy checksum mismatch. Keep the original ZIP for recovery.")
    print("KAGGLE OUTPUT ZIP:", destination.name)
    print("SHA256:", expected)
    print("Download this file from Kaggle's Output panel. Save the completed version with outputs.")
    if display_link:
        from IPython.display import FileLink, display
        # Kaggle's normal notebook cwd is /kaggle/working. The Output panel is
        # the reliable fallback if a frontend does not render a file link.
        if Path.cwd().resolve() == working:
            display(FileLink(destination.name))
    return destination

print("Runtime setup:", configure_runtime(smoke_only=SMOKE_ONLY, require_private_artifacts=False))
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
if not SMOKE_ONLY:
    try:
        print(subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=15))
    except (OSError, subprocess.SubprocessError) as error:
        raise RuntimeError("Select a GPU in Kaggle Session options before running. Setup and model work have not started.") from error
BUNDLE_SHA256 = '71e7e8ca597109387920790117f3071c487ae813796468d4124bc46c2492e2c1'
BUNDLE_BASE64 = (
    '...'
)
def unpack(data, expected_sha256, parent):
    """No pre-existing repo, git executable, or package import is required."""
    if hashlib.sha256(data).hexdigest() != expected_sha256:
        raise ValueError("The embedded notebook bundle is incomplete or changed.")
    root = Path(parent) / ("Lab2_Standalone_" + expected_sha256[:16])
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        if archive.testzip() is not None:
            raise ValueError("The embedded ZIP failed its integrity check.")
        manifest = json.loads(archive.read("STANDALONE_MANIFEST.json"))
        files = manifest["files"]
        if len(archive.namelist()) != len(set(archive.namelist())):
            raise ValueError("Duplicate archive paths are not permitted.")
        if set(archive.namelist()) != set(files) | {"STANDALONE_MANIFEST.json"}:
            raise ValueError("The archive does not match its file manifest.")
        # Check every path and hash before creating or changing any file.
        for member in archive.infolist():
            name = member.filename
            path = PurePosixPath(name)
            if path.is_absolute() or ".." in path.parts or "\\" in name or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Unsafe embedded path.")
            if member.is_dir():
                raise ValueError("Only regular manifest files are permitted.")
            if name != "STANDALONE_MANIFEST.json" and hashlib.sha256(archive.read(name)).hexdigest() != files[name]:
                raise ValueError(f"Embedded source hash mismatch: {name}")
            target = root / name
            if target.is_symlink() or not target.resolve().is_relative_to(root.resolve()):
                raise ValueError("An extraction target escaped the standalone folder.")
            if target.is_file() and target.read_bytes() != archive.read(name):
                raise ValueError(f"Local edits at {target} were preserved. Use a new empty parent folder.")
        root.mkdir(parents=True, exist_ok=True)
        for name in archive.namelist():
            target = root / name
            if not target.is_file():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    return root

parent = Path(os.environ.get("LAB2_STANDALONE_PARENT", "/kaggle/working"))
ROOT = unpack(base64.b64decode(BUNDLE_BASE64), BUNDLE_SHA256, parent)
os.environ["LAB2_STANDALONE_ROOT"] = str(ROOT)
print("Verified embedded source:", BUNDLE_SHA256)
from pathlib import Path
import os, sys, subprocess, json, time

ROOT = Path(os.environ["LAB2_STANDALONE_ROOT"]).resolve()
SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["LAB2_DEVICE"] = "cpu" if SMOKE_ONLY else "cuda"

# This override exists only for a developer CPU smoke, never for a GPU run.
test_python = os.environ.get("LAB2_STANDALONE_TEST_PYTHON", "") if SMOKE_ONLY else ""
if test_python:
    PYTHON = Path(test_python).expanduser().absolute()
    if not PYTHON.is_file():
        raise FileNotFoundError("The explicit CPU test interpreter does not exist.")
    subprocess.run([str(PYTHON), str(ROOT / "tools/preflight.py")], cwd=ROOT, check=True)
    (ROOT / ".lab2_runtime_cpu.json").write_text(json.dumps({"python": str(PYTHON), "cuda_requested": False}))
else:
    subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                    "--device", "cpu" if SMOKE_ONLY else "cuda"], cwd=ROOT, check=True)
    marker = ROOT / (".lab2_runtime_cpu.json" if SMOKE_ONLY else ".lab2_runtime_cuda.json")
    PYTHON = Path(json.loads(marker.read_text())["python"])
SEED = 266
print("Mode:", "CPU SMOKE ONLY: model calls are skipped" if SMOKE_ONLY else "FULL MODEL RUN")
print("Working folder:", ROOT)
print("Pinned interpreter:", PYTHON)

def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    command = [str(PYTHON), "-u", str(ROOT / "tools/monitored_run.py"), "--record", str(record),
               "--", str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f"{Path(relative).name} stopped with exit {status}. Saved log: {log}. Run the final ZIP cell to retain all partial results.")

run_script("tools/standalone_colab.py", "resources")
os.environ["LAB2_RAG_PROFILE"] = "7b-nf4"
os.environ["LAB2_RAG_NATURAL_JSON"] = "1"
os.environ["LAB2_RAG_EVIDENCE_FIRST"] = "1"
os.environ["LAB2_RAG_EVIDENCE_REVIEW"] = "0"
os.environ["LAB2_RAG_REVIEW_REQUIRED"] = "0"


## Run and keep the executed experiment

The isolated child notebook saves after every cell, so its outputs go into the result ZIP. Validation and checkpoint tests run before model generation. Errors are retained and partial results are exported automatically when possible. Unchanged completed answers can resume while this exact run folder still exists.

In [ ]:
try:
    run_script("tools/part1_kaggle.py", "run", *(["--smoke-only"] if SMOKE_ONLY else []))
except Exception:
    import importlib.util
    spec = importlib.util.spec_from_file_location("part1_diagnostics", ROOT / "tools/part1_diagnostics.py")
    diagnostics = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(diagnostics)
    print("\nREAD-ONLY FAILURE DIAGNOSTICS:\n" + diagnostics.failure_details(ROOT))
    print("Keep this session open. The final export cell can be run independently; it does not repeat inference.")
    raise
finally:
    elapsed = time.perf_counter() - NOTEBOOK_STARTED
    timing = dict(scope="Current notebook invocation including setup and experiment; not a sum of prior sessions",
                  elapsed_seconds=elapsed, smoke_only=SMOKE_ONLY,
                  within_90_minutes=(elapsed <= 5400) if not SMOKE_ONLY else None)
    (ROOT / "runtime_checks/notebook_wall_time.json").write_text(json.dumps(timing, indent=2))
    print("NOTEBOOK WALL TIME:", timing)
print((ROOT / "runtime_checks/part1_completion.json").read_text())

## View measured results

No placeholder quality numbers are shown. If running a CPU smoke, model inference and quality measurements are skipped.

In [ ]:
if SMOKE_ONLY:
    print("CPU SMOKE ONLY. No leaderboard CSV or model-quality scores were created.")
elif not (ROOT / "part1_rag/outputs/v2/evaluation/evaluation.json").is_file():
    print("Evaluation is incomplete. No evaluation scores are available yet; no placeholder metrics were created.")
    print("Keep the saved CSV and Results ZIP. Inspect the failed stage above rather than running this display cell again.")
else:
    from IPython.display import display, HTML
    import html
    metrics = json.loads((ROOT / "part1_rag/outputs/v2/evaluation/evaluation.json").read_text())
    columns = ["system", "correct_out_of_25", "retrieval_recall_at_5", "model_judged_citation_accuracy", "median_seconds_per_question", "optional_reviews_completed", "optional_reviews_failed"]
    header = "".join("<th>" + html.escape(key) + "</th>" for key in columns)
    rows = "".join("<tr>" + "".join("<td>" + html.escape(str(row.get(key))) + "</td>" for key in columns) + "</tr>" for row in metrics["results"])
    display(HTML("<table><tr>" + header + "</tr>" + rows + "</table>"))
    print(metrics["limitations"])
    completion = json.loads((ROOT / "runtime_checks/part1_completion.json").read_text())
    print("Experiment elapsed minutes:", round(completion["seconds"] / 60, 2))
    print("Upload THIS new CSV, not the previous 0.419 file:", completion.get("competition_csv"))
    timing = json.loads((ROOT / "runtime_checks/notebook_wall_time.json").read_text())
    print("Notebook elapsed minutes, including setup:", round(timing["elapsed_seconds"] / 60, 2))
    print("Within 90-minute notebook budget:", timing["within_90_minutes"])
    print("Save Version's full runtime also includes the final display/export cells.")

## Download the CSV, executed notebook and evidence

In Kaggle's Output panel download submission_part1_v7.csv (or its bundle-suffixed name if an older CSV was preserved), the Part 1 Results ZIP, and this notebook with its outputs. The exact new CSV path prints above. The ZIP includes the executed experiment, answer traces, evaluation, blank human-rating forms, timing and diagnostic logs. It does not contain pretrained model caches or secrets. Running this cell never repeats model inference.

The completed run still needs real human ratings, team reference review, report/team records, a final tag and the actual Kaggle/Canvas submission. A green computation check does not stand in for those items.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
if "ROOT" not in globals():
    parent = Path(os.environ.get("LAB2_STANDALONE_PARENT", "/kaggle/working"))
    ROOT = parent / "Lab2_Standalone_71e7e8ca59710938"
    if not (ROOT / "tools/part1_kaggle.py").is_file():
        raise FileNotFoundError("This session has no matching saved run. Open the saved version's Output panel; do not restart model work just to export.")
subprocess.run([sys.executable, str(ROOT / "tools/part1_kaggle.py"), "export"], cwd=ROOT, check=True)
print("Download the Lab2_Part1_Results ZIP from Output > /kaggle/working.")